# Лабораторная работа 10: прогнозирование зарплаты по описанию вакансии

## 1. Загрузите данные об описаниях вакансий и соответствующих годовых зарплатах из файла salary-train.csv.

Также загрузите тестовый файл salary-test-mini.csv для последующего прогнозирования.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.feature_extraction import DictVectorizer
import re
from scipy.sparse import hstack

data_train = pd.read_csv('salary-train.csv')
test = pd.read_csv('salary-test-mini.csv')

## 2. Проведите предобработку.

### 2.1 Приведите тексты к нижнему регистру и замените все, кроме букв и цифр, на пробелы.

In [2]:
def pre(text):
    text = re.sub('[^ a−zA−Z0−9]',' ',text.lower())
    return text

data_train['FullDescription'] = data_train['FullDescription'].apply(pre)
test['FullDescription'] = test['FullDescription'].apply(pre)

### 2.2 Примените TfidfVectorizer для преобразования текстов в векторы признаков. Оставьте только слова, встречающиеся хотя бы в 5 объектах (min_df=5).

In [3]:
vec = TfidfVectorizer(min_df = 5)
X_train = vec.fit_transform(data_train['FullDescription'])
X_test = vec.transform(test['FullDescription'])

### 2.3 Замените пропуски в столбцах LocationNormalized и ContractTime на строку 'nan'.

In [4]:
data_train['LocationNormalized'] = data_train['LocationNormalized'].fillna('nan')
data_train['ContractTime'] = data_train['ContractTime'].fillna('nan')
test['LocationNormalized'] = test['LocationNormalized'].fillna('nan')
test['ContractTime'] = test['ContractTime'].fillna('nan')

### 2.4 Примените DictVectorizer для one-hot-кодирования признаков LocationNormalized и ContractTime.

In [5]:
enc = DictVectorizer()
X_train_categ = enc.fit_transform(data_train[['LocationNormalized','ContractTime']].to_dict('records'))
X_test_categ = enc.transform(test[['LocationNormalized','ContractTime']].to_dict('records'))

### 2.5 Объедините все полученные признаки в одну матрицу «объекты-признаки» с помощью scipy.sparse.hstack.

In [6]:
X_train = hstack([X_train, X_train_categ])
X_test = hstack([X_test, X_test_categ])

## 3. Обучите гребневую регрессию с параметром alpha=1. Целевая переменная — SalaryNormalized.

In [7]:
y_train = data_train['SalaryNormalized']
ridge = Ridge(alpha = 1)
ridge.fit(X_train, y_train)

## 4. Постройте прогнозы для двух примеров из файла salary-test-mini.csv. Значения полученных прогнозов являются ответом на задание. Укажите их через пробел (округлив до двух знаков).

In [8]:
ans = ridge.predict(X_test)

with open('1.txt', 'w') as file:
    file.write(' '.join(f"{x:.2f}" for x in ans))